In [0]:
%sql
-- Cria o schema (dataset) 
CREATE SCHEMA IF NOT EXISTS workspace.nyc_taxi

In [0]:
%sql
-- Cria os volumes de armazenamento dos dados brutos e de consumo no schema
CREATE VOLUME IF NOT EXISTS workspace.nyc_taxi.landing_zone;
CREATE VOLUME IF NOT EXISTS workspace.nyc_taxi.consumption_zone;

In [0]:
# Os arquivos foram inseridos na landing zone manualmente
# Verifica os arquivos na landing zone
display(dbutils.fs.ls("/Volumes/workspace/nyc_taxi/landing_zone/"))

In [0]:
# Define os diretórios da landing zone e da consumption zone
LANDING_PATH = "/Volumes/workspace/nyc_taxi/landing_zone/"
CONSUMPTION_PATH = "/Volumes/workspace/nyc_taxi/consumption_zone/"

In [0]:
from functools import reduce
from pyspark.sql import DataFrame
from pyspark.sql.functions import col

# Lê os arquivos parquet da landing zone e define o schema explicitamente das colunas necessárias para que não haja inconsistências nos tipos dos campos

# Lê cada arquivo separadamente
dfs = []
for month in ["01", "02", "03", "04", "05"]:
    path = f"{LANDING_PATH}yellow_tripdata_2023-{month}.parquet"
    df = spark.read.parquet(path)

    # Renomeia Airport_fee para airport_fee se necessário
    if "Airport_fee" in df.columns:
        df = df.withColumnRenamed("Airport_fee", "airport_fee")

    # Seleciona e casteia as colunas obrigatórias com os tipos corretos
    df = df.select(
        col("VendorID").cast("long"),
        col("passenger_count").cast("integer"),
        col("total_amount").cast("double"),
        col("tpep_pickup_datetime").cast("timestamp"),
        col("tpep_dropoff_datetime").cast("timestamp"),
    )
    dfs.append(df)

# Une todos os meses em um único DataFrame
df_final = reduce(DataFrame.union, dfs)

print(f"Total de linhas: {df_final.count():,}")
df_final.printSchema()

In [0]:
# Salvando os dados em uma tabela delta no catálogo
# Salva os dados na camada de consumo como Delta Table
df_final.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.nyc_taxi.yellow_trips")

In [0]:
%sql
-- Consultando a tabela para validacão
SELECT * FROM workspace.nyc_taxi.yellow_trips LIMIT 10